In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import gc
import glob
import re
import xarray as xr
import numpy as np
import pandas as pd
from multiprocessing import Pool
from eofs.xarray import Eof as XarrayEof
from eofs.standard import Eof as StandardEof

# ================= 1. 核心配置区 =================
CORES = 10
OVERWRITE = False

LAT_MIN = 20.0
LAT_MAX = 90.0
AO_PLEV_HPA = 1000.0

TARGET_PLEV_HPA = np.array([
    1000.0, 950.0, 900.0, 850.0, 800.0, 750.0,
    700.0, 600.0, 550.0, 500.0, 450.0, 400.0,
    350.0, 300.0, 250.0, 225.0, 200.0, 175.0,
    150.0, 125.0, 100.0, 70.0, 50.0, 30.0,
    20.0, 10.0, 7.0, 5.0, 3.0, 2.0, 1.0
], dtype=np.float64)
TARGET_PLEV_PA = TARGET_PLEV_HPA * 100.0

# AO 使用 Code-B modified-weights 方法时的目标纬度
AO_TARGET_LAT = np.arange(20.0, 90.0 + 0.001, 2.5)

# ---------------- 实验配置 ----------------
# 现在：
#   - B2000WCN、NOCOUPL 与 Clim3D 读 *_timefixed
#   - BWCN 直接读本目录
#
# use_reference:
#   False = 自己训练 EOF / 气候态
#   True  = 使用 reference 组（通常是 coupled）的 EOF / 气候态来投影
EXPERIMENTS = [
    {
        "name": "B2000WCN001002",
        "base_dir": "/mnt/soclim0/public_data/weiji/B2000WCN001002_timefixed",
        "file_template": "B2000WCN.sample.cam.h3.{year:04d}.{var}.nc",
        "is_reference": True,
        "use_reference": False
    },
    {
        "name": "B2000WCN_NOCOUPL001002",
        "base_dir": "/mnt/soclim0/public_data/weiji/B2000WCN_NOCOUPL001002_timefixed",
        "file_template": "B2000WCN.NOCOUPL.sample.cam.h3.{year:04d}.{var}.nc",
        "is_reference": False,
        "use_reference": False
    },
    {
        "name": "B2000WCN007009010011_Clim3D",
        "base_dir": "/mnt/soclim0/public_data/weiji/B2000WCN007009010011_Clim3D_timefixed",
        "file_template": "B2000WCN.NOCOUPL.sample.cam.h3.{year:04d}.{var}.nc",
        "is_reference": False,
        "use_reference": False
    },
    {
        "name": "BWCN",
        "base_dir": "/mnt/soclim0/public_data/weiji/BWCN",
        "file_template": "BWCN.cam.h3.{year:04d}.{var}.nc",
        "is_reference": False,
        "use_reference": True
    }
]

REF_DATA = {}
YEAR_RE = re.compile(r"\.cam\.h3\.(\d{4})\.")

# ================= 2. 基础工具函数 =================
def parse_year_from_filename(path: str) -> int:
    m = YEAR_RE.search(os.path.basename(path))
    if not m:
        raise ValueError(f"Cannot parse year from filename: {path}")
    return int(m.group(1))

def list_available_years(base_dir):
    z3_dir = os.path.join(base_dir, "Z3")
    z3_files = sorted(glob.glob(os.path.join(z3_dir, "*.Z3.nc")))
    if not z3_files:
        z3_files = sorted(glob.glob(os.path.join(base_dir, "*.Z3.nc")))

    years = []
    for f in z3_files:
        try:
            years.append(parse_year_from_filename(f))
        except Exception:
            pass

    return sorted(set(years))

def output_paths(data_out_dir, exp_name):
    return {
        "ao_nc":  os.path.join(data_out_dir, f"{exp_name}_Daily_AO_Index.nc"),
        "ao_csv": os.path.join(data_out_dir, f"{exp_name}_Daily_AO_Index.csv"),
        "nam_nc": os.path.join(data_out_dir, f"{exp_name}_Vertical_NAM.nc"),
    }

def outputs_exist(data_out_dir, exp_name):
    outs = output_paths(data_out_dir, exp_name)
    return (
        os.path.exists(outs["ao_nc"])  and os.path.getsize(outs["ao_nc"])  > 0 and
        os.path.exists(outs["ao_csv"]) and os.path.getsize(outs["ao_csv"]) > 0 and
        os.path.exists(outs["nam_nc"]) and os.path.getsize(outs["nam_nc"]) > 0
    )

def _process_wrapper(args):
    return process_one_year(*args)

def logp_interp_extrap_1d(p_prof, z_prof, target_p):
    p = np.asarray(p_prof, dtype=np.float64)
    z = np.asarray(z_prof, dtype=np.float64)
    tp = np.asarray(target_p, dtype=np.float64)

    mask = np.isfinite(p) & np.isfinite(z) & (p > 0)
    p = p[mask]
    z = z[mask]

    if p.size < 2:
        return np.full(tp.shape, np.nan, dtype=np.float64)

    order = np.argsort(p)
    p = p[order]
    z = z[order]

    lp = np.log(p)
    ltp = np.log(tp)

    out = np.interp(ltp, lp, z)

    mask_top = ltp < lp[0]
    if np.any(mask_top):
        slope_top = (z[1] - z[0]) / (lp[1] - lp[0])
        out[mask_top] = z[0] + slope_top * (ltp[mask_top] - lp[0])

    mask_bottom = ltp > lp[-1]
    if np.any(mask_bottom):
        slope_bottom = (z[-1] - z[-2]) / (lp[-1] - lp[-2])
        out[mask_bottom] = z[-1] + slope_bottom * (ltp[mask_bottom] - lp[-1])

    return out

def interp_to_pressure_levels(z3_da, p3d_da, target_p_pa, target_p_hpa):
    z_on_plev = xr.apply_ufunc(
        logp_interp_extrap_1d,
        p3d_da,
        z3_da,
        input_core_dims=[["lev"], ["lev"]],
        output_core_dims=[["plev"]],
        vectorize=True,
        dask="parallelized",
        kwargs={"target_p": target_p_pa},
        output_dtypes=[np.float64],
        dask_gufunc_kwargs={"output_sizes": {"plev": len(target_p_pa)}},
    )

    z_on_plev = z_on_plev.assign_coords(plev=("plev", target_p_hpa))
    z_on_plev["plev"].attrs["units"] = "hPa"
    z_on_plev["plev"].attrs["long_name"] = "pressure"

    z_on_plev = z_on_plev.transpose("time", "plev", "lat", "lon")
    z_on_plev = z_on_plev.rename({"plev": "lev"})
    z_on_plev["lev"].attrs["units"] = "hPa"
    z_on_plev["lev"].attrs["long_name"] = "pressure"

    return z_on_plev

def get_valid_time_mask(da):
    """
    对任意 DataArray，返回一个 time 维的布尔掩码：
    只有当该时刻在所有非 time 维度上都是有限值时，才为 True。
    这样填充出来的全 NaN 日期会被跳过，计算结果再恢复为 NaN。
    """
    if "time" not in da.dims:
        raise ValueError("Input DataArray has no 'time' dimension.")

    spatial_dims = [d for d in da.dims if d != "time"]
    valid = xr.apply_ufunc(np.isfinite, da)
    for d in spatial_dims:
        valid = valid.all(d)
    return valid

def print_nan_filter_summary(da, valid_time_mask, label):
    """打印被 NaN/inf 过滤掉的时间类型，帮助区分全 NaN 填充日和局部缺测。"""
    spatial_dims = [d for d in da.dims if d != "time"]
    finite = xr.apply_ufunc(np.isfinite, da)
    all_missing = ~finite
    for d in spatial_dims:
        all_missing = all_missing.all(d)

    n_invalid = int((~valid_time_mask).sum().values)
    n_all_missing = int(all_missing.sum().values)
    n_partial_missing = n_invalid - n_all_missing
    print(
        f"--> {label} invalid detail: all-NaN/inf times={n_all_missing}, "
        f"partial-NaN/inf times={n_partial_missing}"
    )

def restore_to_full_time(valid_da, full_time, name=None, attrs=None):
    """
    把只在有效时刻上计算出来的结果，恢复到完整 full_time 上。
    缺失时刻自动填 NaN。
    支持:
      - 1D: time
      - 2D: time x lev
    """
    if valid_da.ndim == 1:
        out = xr.DataArray(
            np.full((len(full_time),), np.nan, dtype=np.float64),
            coords={"time": full_time},
            dims=("time",),
            name=name if name is not None else valid_da.name
        )
        out.loc[dict(time=valid_da.time)] = valid_da.values

    elif valid_da.ndim == 2 and ("time" in valid_da.dims):
        other_dim = [d for d in valid_da.dims if d != "time"][0]
        out = xr.DataArray(
            np.full((len(full_time), valid_da.sizes[other_dim]), np.nan, dtype=np.float64),
            coords={"time": full_time, other_dim: valid_da[other_dim].values},
            dims=("time", other_dim),
            name=name if name is not None else valid_da.name
        )
        out.loc[dict(time=valid_da.time)] = valid_da.values

    else:
        raise ValueError(f"Unsupported ndim/dims for restore_to_full_time: {valid_da.dims}")

    if attrs is not None:
        out.attrs = attrs

    return out

def process_one_year(year_int, base_dir, file_template):
    f_z3_name = file_template.format(year=year_int, var="Z3")
    f_ps_name = file_template.format(year=year_int, var="PS")

    f_z3 = os.path.join(base_dir, "Z3", f_z3_name)
    if not os.path.exists(f_z3):
        f_z3 = os.path.join(base_dir, f_z3_name)

    f_ps = os.path.join(base_dir, "PS", f_ps_name)
    if not os.path.exists(f_ps):
        f_ps = os.path.join(base_dir, f_ps_name)

    if not (os.path.exists(f_z3) and os.path.exists(f_ps)):
        return None

    try:
        time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)

        with xr.open_dataset(f_z3, decode_times=time_coder) as ds_z, \
             xr.open_dataset(f_ps, decode_times=time_coder) as ds_ps:

            p0 = 100000.0
            hyam = ds_z["hyam"]
            hybm = ds_z["hybm"]
            ps = ds_ps["PS"]

            p3d = hyam * p0 + hybm * ps

            # 北半球 20N–90N
            z3_nh = ds_z["Z3"].sel(lat=slice(LAT_MIN, LAT_MAX))
            p3d_nh = p3d.sel(lat=slice(LAT_MIN, LAT_MAX))

            z3_plev_nh = interp_to_pressure_levels(
                z3_da=z3_nh,
                p3d_da=p3d_nh,
                target_p_pa=TARGET_PLEV_PA,
                target_p_hpa=TARGET_PLEV_HPA
            )

            # AO: 1000 hPa + zonal mean
            da_z3_1000_zm = z3_plev_nh.sel(lev=AO_PLEV_HPA).mean("lon")
            da_z3_1000_zm.name = "Z3_1000_ZM"

            # NAM: full vertical zonal mean
            z3_zm = z3_plev_nh.mean("lon")
            z3_zm.name = "Z3_ZM"

            out_1000 = da_z3_1000_zm.compute()
            out_1000.name = "Z3_1000_ZM"

            out_zm = z3_zm.compute()
            out_zm.name = "Z3_ZM"
            out_zm["lev"].attrs["units"] = "hPa"
            out_zm["lev"].attrs["long_name"] = "pressure"

            return (out_1000, out_zm)

    except Exception as e:
        print(f"Error in {base_dir} year {year_int:04d}: {str(e)}")
        return None

# ================= 3. AO: Code-B modified weights (zonal mean) =================
def prepare_ao_field_codeB(z3_1000_zm):
    # 输入: time x lat。WACCM/CAM 网格在 20N 附近常从 21.789N 开始；
    # 允许很小的边界外推，避免目标 20N 整列 NaN 把所有日期误删。
    da = z3_1000_zm.sortby("lat")
    da = da.interp(lat=AO_TARGET_LAT, kwargs={"fill_value": "extrapolate"})
    da = da.assign_coords(lat=("lat", AO_TARGET_LAT))
    da.name = "Z3_1000_ZM_interp"
    return da

def decide_flip_codeB(gh_layer, ao_raw):
    """
    尽量贴近你给的 Code-B 思路：
    比较正 AO 最大时刻，较低纬半边 vs 较高纬半边。
    如果高纬更高（即 left-right < 0），则翻转。
    """
    nof_lats = gh_layer.shape[1]
    max_AO = int(np.nanargmax(ao_raw))

    i0 = 0
    i1 = max(1, int(nof_lats / 2) - 1)
    j0 = min(int(nof_lats / 2) + 1, nof_lats - 1)
    j1 = nof_lats

    left_mean = np.nanmean(gh_layer[max_AO, i0:i1])
    right_mean = np.nanmean(gh_layer[max_AO, j0:j1])

    flip = -1 if (left_mean - right_mean) < 0 else 1
    return flip

def compute_ao_codeB_modified(z3_1000_zm_all, exp, exp_name):
    """
    返回:
      ao_index_daily_full : DataArray(time)
    计算时只使用有效时间样本；
    返回时恢复到完整 time 轴，缺失时刻填 NaN。
    """
    z_ao_full = prepare_ao_field_codeB(z3_1000_zm_all)

    # 先筛掉含 NaN 的时间样本
    valid_time_mask = get_valid_time_mask(z_ao_full)
    n_valid = int(valid_time_mask.sum().values)
    n_total = z_ao_full.sizes["time"]
    n_drop = n_total - n_valid
    print(f"--> AO valid times: {n_valid} / {n_total} (drop {n_drop})")
    print_nan_filter_summary(z_ao_full, valid_time_mask, "AO")

    z_ao = z_ao_full.sel(time=valid_time_mask)

    if z_ao.sizes["time"] == 0:
        raise ValueError(f"{exp_name}: AO input has no valid time samples after NaN filtering.")

    if not exp["use_reference"]:
        # 自己训练
        clim_doy = z_ao.groupby("time.dayofyear").mean("time")
        ao_anom = z_ao.groupby("time.dayofyear") - clim_doy
        if "dayofyear" in ao_anom.coords:
            ao_anom = ao_anom.drop_vars("dayofyear")

        gh_layer = np.array(ao_anom, dtype=np.float64)   # time x lat
        lats_np = ao_anom["lat"].values.astype(np.float64)

        coslat = np.cos(np.deg2rad(lats_np)).clip(0.0, 1.0)
        wgts_mod = np.sqrt(coslat)

        solver = StandardEof(gh_layer, weights=wgts_mod, center=True)

        ao_raw = np.reshape(solver.pcs(npcs=1, pcscaling=1), (gh_layer.shape[0],))
        flip = decide_flip_codeB(gh_layer, ao_raw)
        ao_vals = ao_raw * flip

        ao_index_valid = xr.DataArray(
            ao_vals,
            coords={"time": ao_anom["time"].values},
            dims=["time"],
            name="AO_Index"
        )

        if exp["is_reference"]:
            REF_DATA["AO_clim_doy"] = clim_doy
            REF_DATA["AO_solver"] = solver
            REF_DATA["AO_flip"] = flip
            REF_DATA["AO_lat_target"] = lats_np
            print(f"--> [记录] {exp_name} 的 AO 参考模态已缓存。")

    else:
        # 用 B2000WCN reference 投影
        print(f"--> [核心] {exp_name} AO 使用参考组 (B2000WCN001002) 的 Code-B modified 模态投影。")

        clim_doy = REF_DATA["AO_clim_doy"]
        solver = REF_DATA["AO_solver"]
        flip = REF_DATA["AO_flip"]
        lat_target = REF_DATA["AO_lat_target"]

        z_ao = z_ao.interp(lat=lat_target)
        z_ao = z_ao.assign_coords(lat=("lat", lat_target))

        ao_anom = z_ao.groupby("time.dayofyear") - clim_doy
        if "dayofyear" in ao_anom.coords:
            ao_anom = ao_anom.drop_vars("dayofyear")

        gh_layer = np.array(ao_anom, dtype=np.float64)

        # projectField + eofscaling=1 => 与 pcs(..., pcscaling=1) 同量纲
        ao_proj = solver.projectField(gh_layer, neofs=1, eofscaling=1, weighted=True)
        ao_proj = np.reshape(np.asarray(ao_proj), (gh_layer.shape[0],))
        ao_vals = ao_proj * flip

        ao_index_valid = xr.DataArray(
            ao_vals,
            coords={"time": ao_anom["time"].values},
            dims=["time"],
            name="AO_Index"
        )

    ao_index_valid.attrs["long_name"] = f"Daily AO index ({exp_name}, Code-B modified weights, zonal mean)"

    # 恢复到完整时间轴：无效时刻填 NaN
    ao_index_daily_full = restore_to_full_time(
        ao_index_valid,
        full_time=z_ao_full.time.values,
        name="AO_Index",
        attrs=ao_index_valid.attrs
    )

    return ao_index_daily_full

# ================= 4. NAM: zonal-mean vertical EOF projection =================
def compute_nam_vertical(z3_zm_all, exp, exp_name):
    """
    返回:
      nam_vertical_full : DataArray(time, lev)
    计算时只使用有效时间样本；
    返回时恢复到完整 time 轴，缺失时刻填 NaN。
    """
    # 先筛掉含 NaN 的时间样本
    valid_time_mask = get_valid_time_mask(z3_zm_all)
    n_valid = int(valid_time_mask.sum().values)
    n_total = z3_zm_all.sizes["time"]
    n_drop = n_total - n_valid
    print(f"--> NAM valid times: {n_valid} / {n_total} (drop {n_drop})")
    print_nan_filter_summary(z3_zm_all, valid_time_mask, "NAM")

    z3_zm_valid = z3_zm_all.sel(time=valid_time_mask)

    if z3_zm_valid.sizes["time"] == 0:
        raise ValueError(f"{exp_name}: NAM input has no valid time samples after NaN filtering.")

    nam_daily_levels = []

    if not exp["use_reference"]:
        # 1. 月均异常用于训练 EOF
        z3_zm_monthly = z3_zm_valid.resample(time="1MS").mean().dropna(dim="time", how="all")
        clim_mon_zm = z3_zm_monthly.groupby("time.month").mean("time")
        anom_mon_zm = z3_zm_monthly.groupby("time.month") - clim_mon_zm

        # 2. DOY 平滑异常用于 daily projection
        clim_doy_zm = z3_zm_valid.groupby("time.dayofyear").mean("time")
        clim_doy_zm_smooth = clim_doy_zm.rolling(
            dayofyear=21,
            center=True,
            min_periods=1,
        ).mean()
        anom_doy_zm = z3_zm_valid.groupby("time.dayofyear") - clim_doy_zm_smooth
        if "dayofyear" in anom_doy_zm.coords:
            anom_doy_zm = anom_doy_zm.drop_vars("dayofyear")

        coslat_da = np.clip(np.cos(np.deg2rad(z3_zm_valid.lat)), 0, None)
        wgts_1d = np.sqrt(coslat_da)

        nam_ref_data_levels = {}

        print("--> 正在逐层进行 NAM EOF 分解与每日投影...")
        for lev in z3_zm_valid.lev.values:
            da_mon = anom_mon_zm.sel(lev=lev)
            da_day = anom_doy_zm.sel(lev=lev)

            solver_nam = XarrayEof(da_mon, weights=wgts_1d.values)
            eof1_nam = solver_nam.eofs(neofs=1, eofscaling=0).squeeze()
            pc1_raw_nam = solver_nam.pcs(npcs=1, pcscaling=0).squeeze()

            if float(eof1_nam.sel(lat=80.0, method="nearest").values) > 0:
                flip_nam = -1
            else:
                flip_nam = 1

            pc1_raw_nam = pc1_raw_nam * flip_nam
            pc_mean = pc1_raw_nam.mean(dim="time")
            pc_std = pc1_raw_nam.std(dim="time")

            nam_ref_data_levels[float(lev)] = {
                "solver": solver_nam,
                "flip_factor": flip_nam,
                "pc_mean": pc_mean,
                "pc_std": pc_std
            }

            pseudo_pcs_nam = solver_nam.projectField(da_day, neofs=1, eofscaling=0).squeeze() * flip_nam
            nam_day = (pseudo_pcs_nam - pc_mean) / pc_std
            nam_day = nam_day.assign_coords(lev=float(lev)).expand_dims("lev")
            nam_daily_levels.append(nam_day)

        if exp["is_reference"]:
            REF_DATA["NAM_clim_doy_smooth"] = clim_doy_zm_smooth
            REF_DATA["NAM_levels"] = nam_ref_data_levels
            print(f"--> [记录] {exp_name} 的 NAM 参考模态已缓存。")

    else:
        print(f"--> [核心] {exp_name} NAM 使用参考组 (B2000WCN001002) 的逐层 EOF 模态投影。")

        clim_doy_zm_smooth = REF_DATA["NAM_clim_doy_smooth"]
        nam_ref_data_levels = REF_DATA["NAM_levels"]

        anom_doy_zm = z3_zm_valid.groupby("time.dayofyear") - clim_doy_zm_smooth
        if "dayofyear" in anom_doy_zm.coords:
            anom_doy_zm = anom_doy_zm.drop_vars("dayofyear")

        for lev in z3_zm_valid.lev.values:
            da_day = anom_doy_zm.sel(lev=lev)

            ref_dict = nam_ref_data_levels[float(lev)]
            solver_nam = ref_dict["solver"]
            flip_nam = ref_dict["flip_factor"]
            pc_mean = ref_dict["pc_mean"]
            pc_std = ref_dict["pc_std"]

            pseudo_pcs_nam = solver_nam.projectField(da_day, neofs=1, eofscaling=0).squeeze() * flip_nam
            nam_day = (pseudo_pcs_nam - pc_mean) / pc_std
            nam_day = nam_day.assign_coords(lev=float(lev)).expand_dims("lev")
            nam_daily_levels.append(nam_day)

    nam_vertical_valid = xr.concat(nam_daily_levels, dim="lev").transpose("time", "lev")
    nam_vertical_valid.name = "NAM_Vertical"
    nam_vertical_valid.attrs["long_name"] = f"EOF-based Vertical NAM index ({exp_name})"
    nam_vertical_valid["lev"].attrs["units"] = "hPa"
    nam_vertical_valid["lev"].attrs["long_name"] = "pressure"

    # 恢复到完整时间轴：无效时刻填 NaN
    nam_vertical_full = restore_to_full_time(
        nam_vertical_valid,
        full_time=z3_zm_all.time.values,
        name="NAM_Vertical",
        attrs=nam_vertical_valid.attrs
    )
    nam_vertical_full["lev"].attrs["units"] = "hPa"
    nam_vertical_full["lev"].attrs["long_name"] = "pressure"

    return nam_vertical_full

# ================= 5. 主循环 =================
if __name__ == "__main__":
    for exp in EXPERIMENTS:
        exp_name = exp["name"]
        base_dir = exp["base_dir"]
        file_template = exp["file_template"]

        data_out_dir = os.path.join(base_dir, "NAM")
        os.makedirs(data_out_dir, exist_ok=True)

        outs = output_paths(data_out_dir, exp_name)
        already_done = outputs_exist(data_out_dir, exp_name)

        print("\n" + "🚀" * 25)
        print(f"正在处理实验组: {exp_name}")
        print(f"base_dir = {base_dir}")
        print("🚀" * 25)

        # 非 reference 组：如果已有输出且不覆写，直接跳过
        # reference 组：即使已有输出，也仍重新算一遍，以便 BWCN 可投影到参考模态
        if already_done and (not OVERWRITE) and (not exp["is_reference"]):
            print(f"⏭️ 输出已存在，跳过: {exp_name}")
            continue

        years_to_process = list_available_years(base_dir)
        if not years_to_process:
            print(f"❌ 警告：未找到 {exp_name} 的有效 Z3 文件，跳过该组。")
            continue

        print(f"--> 实际扫描到 {len(years_to_process)} 年: {years_to_process[0]}–{years_to_process[-1]}")
        args_list = [(y, base_dir, file_template) for y in years_to_process]

        with Pool(processes=CORES) as pool:
            results = pool.map(_process_wrapper, args_list)

        valid_results = [res for res in results if res is not None]
        if not valid_results:
            print(f"❌ 警告：未找到 {exp_name} 的有效数据，跳过该组。")
            continue

        # AO input: time x lat (1000 hPa zonal mean)
        z3_1000_zm_all = xr.concat([res[0] for res in valid_results], dim="time").sortby("time")

        # NAM input: time x lev x lat (zonal mean)
        z3_zm_all = xr.concat([res[1] for res in valid_results], dim="time").sortby("time")
        z3_zm_all["lev"].attrs["units"] = "hPa"
        z3_zm_all["lev"].attrs["long_name"] = "pressure"

        # 去重（保险）
        _, idx1000 = np.unique(z3_1000_zm_all["time"].values, return_index=True)
        z3_1000_zm_all = z3_1000_zm_all.isel(time=np.sort(idx1000))

        _, idxzm = np.unique(z3_zm_all["time"].values, return_index=True)
        z3_zm_all = z3_zm_all.isel(time=np.sort(idxzm))

        del results, valid_results
        gc.collect()

        # ---------------- AO ----------------
        print(f"🌟 开始处理 {exp_name} 的 AO 指数 (Code-B modified, zonal mean)...")
        ao_index_daily = compute_ao_codeB_modified(z3_1000_zm_all, exp, exp_name)

        # ---------------- NAM ----------------
        print(f"🌟 开始处理 {exp_name} 的多层垂直 NAM 指数...")
        nam_vertical = compute_nam_vertical(z3_zm_all, exp, exp_name)

        # ---------------- 保存 ----------------
        # reference 组如果已有输出且不覆写，则不重写文件，但已经完成了 reference 缓存
        if already_done and (not OVERWRITE) and exp["is_reference"]:
            print(f"⏭️ {exp_name} 已有输出，参考模态已刷新到内存，但不覆写磁盘文件。")
        else:
            ao_index_daily.to_netcdf(outs["ao_nc"])
            pd.DataFrame({
                "Date": ao_index_daily.time.values,
                "AO_Index": ao_index_daily.values
            }).to_csv(outs["ao_csv"], index=False)

            nam_vertical.to_netcdf(outs["nam_nc"])

            print(f"✅ {exp_name} AO 已保存:")
            print(f"   NC : {outs['ao_nc']}")
            print(f"   CSV: {outs['ao_csv']}")
            print(f"✅ {exp_name} NAM 已保存:")
            print(f"   NC : {outs['nam_nc']}")

        del z3_1000_zm_all, z3_zm_all, ao_index_daily, nam_vertical
        gc.collect()

    print("\n🎉 全部实验 AO / NAM 计算完成！")


🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀
正在处理实验组: B2000WCN001002
base_dir = /mnt/soclim0/public_data/weiji/B2000WCN001002_timefixed
🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀
--> 实际扫描到 210 年: 1–210
🌟 开始处理 B2000WCN001002 的 AO 指数 (Code-B modified, zonal mean)...
--> AO valid times: 75850 / 76650 (drop 800)
--> AO invalid detail: all-NaN/inf times=800, partial-NaN/inf times=0
--> [记录] B2000WCN001002 的 AO 参考模态已缓存。
🌟 开始处理 B2000WCN001002 的多层垂直 NAM 指数...
--> NAM valid times: 75850 / 76650 (drop 800)
--> NAM invalid detail: all-NaN/inf times=800, partial-NaN/inf times=0
--> 正在逐层进行 NAM EOF 分解与每日投影...
--> [记录] B2000WCN001002 的 NAM 参考模态已缓存。
⏭️ B2000WCN001002 已有输出，参考模态已刷新到内存，但不覆写磁盘文件。

🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀
正在处理实验组: B2000WCN_NOCOUPL001002
base_dir = /mnt/soclim0/public_data/weiji/B2000WCN_NOCOUPL001002_timefixed
🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀
⏭️ 输出已存在，跳过: B2000WCN_NOCOUPL001002

🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀
正在处理实验组: B2000WCN007009010011_Clim3D
base_dir = /mnt/soclim0/public_data/weiji/B2000WCN007009010011_Clim3D_timefixed
🚀🚀🚀🚀🚀🚀🚀🚀🚀